# Task 6: Generate evaluation outputs

**Settings:** Accelerator **GPU T4 x2**, Internet **on**.
**Inputs:** Add Input > Your Work > the saved outputs of **pa2-03-sft**, **pa2-04-dpo** and **pa2-05-lora-persona**.
Run with **Save Version > Save & Run All (Commit)**.

Three comparisons, each on fixed prompts with identical decoding for both sides
(greedy, repetition penalty 1.1, at most 512 new tokens, fp32; chosen on held-out prompts in Task 3):

| Comparison | Prompts | Evaluated model | Baseline |
|---|---|---|---|
| `dpo_hh_vs_sft` | `eval_sets/hh_rlhf_eval_300_seed42.json` (300) | DPO-HH best checkpoint | SFT best checkpoint |
| `dpo_ultrafeedback_vs_sft` | `eval_sets/alpaca_eval_300_seed42.json` (300) | DPO-UltraFeedback best checkpoint | SFT best checkpoint |
| `lora_vs_base_persona` | `eval_sets/persona_style_eval_40.json` (40) | SmolLM2-360M-Instruct + LoRA-DPO adapter | SmolLM2-360M-Instruct |

A sanity check (first 3 prompts per model) runs before the full generation. Then blind pair files are built:
for each prompt a seeded coin flip decides which answer is Response A, and the model identities are stored only
under `hidden`, which the judge never sees (`src/pa2/pairs.py`). Everything is archived as `task6_generation_outputs.zip`.

In [ ]:
REPO_URL = "https://github.com/mabdullahi7780/small-model-alignment-starter.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working"
SCRATCH_DIR = "/tmp"
REPO_DIR = f"{SCRATCH_DIR}/small-model-alignment-starter"
OUT_DIR = f"{WORK_DIR}/outputs/generate"
QWEN_TOKENIZER = "Qwen/Qwen2.5-0.5B"
SMOL = "HuggingFaceTB/SmolLM2-360M-Instruct"
HH = "eval_sets/hh_rlhf_eval_300_seed42.json"
ALPACA = "eval_sets/alpaca_eval_300_seed42.json"
PERSONA = "eval_sets/persona_style_eval_40.json"
DECODING = ["--max-new-tokens", "512", "--repetition-penalty", "1.1", "--batch-size", "32", "--dtype", "float32"]
SANITY_PROMPTS = 3
RUN_SETTINGS = {"decoding": DECODING, "sanity_prompts": SANITY_PROMPTS, "qwen_tokenizer": QWEN_TOKENIZER, "smol": SMOL}

In [ ]:
import os
import subprocess
import sys


def run(cmd, cwd=None, env=None, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)


if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=commit %H%nauthor date %ad%nsubject %s"], cwd=REPO_DIR)

In [ ]:
# Kaggle's preinstalled torch is built for its GPU runtime; pin it so pip can never replace it.
TORCH_BEFORE = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("Preinstalled torch:", TORCH_BEFORE)

constraint_file = f"{SCRATCH_DIR}/torch-constraint.txt"
with open(constraint_file, "w") as handle:
    handle.write(f"torch=={TORCH_BEFORE}\n")

run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-c", constraint_file], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)

In [ ]:
import json

COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
os.makedirs(f"{OUT_DIR}/logs", exist_ok=True)
with open(f"{OUT_DIR}/run_info.json", "w") as handle:
    json.dump({"commit": COMMIT, "torch": TORCH_BEFORE, **RUN_SETTINGS}, handle, indent=2)
print("commit", COMMIT)

In [ ]:
def run_on_gpus(jobs, tag):
    """Start one subprocess per job, each pinned to its own GPU, and wait for all of them.

    jobs: list of (name, gpu_index, argv). Each job's output goes to logs/<tag>_<name>.log.
    """
    # expandable_segments lets the CUDA allocator reuse freed memory instead of fragmenting it.
    env_base = {**os.environ, "PYTHONPATH": f"{REPO_DIR}/src:{REPO_DIR}", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    running = []
    for name, gpu, argv in jobs:
        log_path = f"{OUT_DIR}/logs/{tag}_{name}.log"
        handle = open(log_path, "w")
        print(f"start {tag}/{name} on GPU {gpu}", flush=True)
        process = subprocess.Popen(
            [sys.executable, *argv], cwd=REPO_DIR, stdout=handle, stderr=subprocess.STDOUT,
            env={**env_base, "CUDA_VISIBLE_DEVICES": str(gpu)},
        )
        running.append((name, process, handle, log_path))
    failed = []
    for name, process, handle, log_path in running:
        code = process.wait()
        handle.close()
        print(f"\n===== {tag}/{name}: exit code {code} =====")
        print(open(log_path).read()[-8000:])
        if code:
            failed.append(name)
    if failed:
        raise RuntimeError(f"{tag}: failed jobs {failed}; see {OUT_DIR}/logs")

## The attached checkpoints

In [ ]:
import glob


def find(pattern, what):
    found = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not found:
        raise RuntimeError(f"{what} not found: attach the notebook output that contains {pattern}")
    return os.path.dirname(found[0])


SFT = find("outputs/sft/run/best/model.safetensors", "SFT best checkpoint")
DPO_HH = find("outputs/dpo/hh/run/best/model.safetensors", "DPO-HH best checkpoint")
DPO_UF = find("outputs/dpo/ultrafeedback/run/best/model.safetensors", "DPO-UltraFeedback best checkpoint")
LORA = find("outputs/lora/run/adapter/adapter_model.safetensors", "LoRA adapter")
for name, path in (("SFT", SFT), ("DPO-HH", DPO_HH), ("DPO-UltraFeedback", DPO_UF)):
    info = json.load(open(f"{path}/checkpoint_info.json"))
    print(f"{name:<18} step {info.get('step')}  {path}")
print(f"{'LoRA adapter':<18} {LORA}")

## Generation jobs

Two queues, one per GPU; each round runs one job on each GPU at the same time.

In [ ]:
def qwen_job(name, model, prompts, generator):
    return name, ["--model", model, "--tokenizer", QWEN_TOKENIZER, "--generator", generator, "--prompts", prompts]


def smol_job(name, generator, adapter=None):
    args = ["--model", SMOL, "--prompt-format", "chat_template", "--generator", generator, "--prompts", PERSONA]
    return name, args + (["--adapter", adapter] if adapter else [])


GPU0 = [qwen_job("sft_alpaca", SFT, ALPACA, "qwen2.5-0.5b-sft"),
        qwen_job("dpo_ultrafeedback_alpaca", DPO_UF, ALPACA, "qwen2.5-0.5b-dpo-ultrafeedback"),
        smol_job("smollm2_base_persona", "smollm2-360m-instruct")]
GPU1 = [qwen_job("sft_hh", SFT, HH, "qwen2.5-0.5b-sft"),
        qwen_job("dpo_hh_hh", DPO_HH, HH, "qwen2.5-0.5b-dpo-hh"),
        smol_job("smollm2_lora_persona", "smollm2-360m-instruct-lora-dpo", LORA)]


def run_rounds(tag, out_dir, extra):
    for (name0, args0), (name1, args1) in zip(GPU0, GPU1):
        run_on_gpus([
            (name0, 0, ["-m", "pa2.generate", *args0, "--out", f"{out_dir}/{name0}.json", *DECODING, *extra]),
            (name1, 1, ["-m", "pa2.generate", *args1, "--out", f"{out_dir}/{name1}.json", *DECODING, *extra]),
        ], tag)

## Sanity check: the first 3 prompts per model

Read these before trusting the full run: right checkpoint, prompt format, stopping with EOS, sensible length.

In [ ]:
run_rounds("sanity", f"{OUT_DIR}/sanity", ["--limit", str(SANITY_PROMPTS), "--show", str(SANITY_PROMPTS)])
for path in sorted(glob.glob(f"{OUT_DIR}/sanity/*.json")):
    if path.endswith(".meta.json"):
        continue
    rows = json.load(open(path))
    if not any(row["output"] for row in rows):
        raise RuntimeError(f"{path}: every sanity answer is empty")
    print(f"{os.path.basename(path):<36}", [(row["stop_reason"], row["new_tokens"]) for row in rows])

## Full generation: 1,280 answers

In [ ]:
run_rounds("answers", f"{OUT_DIR}/answers", ["--skip-existing", "--show", "1"])

## Blind A/B pair files

In [ ]:
ANSWERS = f"{OUT_DIR}/answers"
COMPARISONS = [
    ("dpo_hh_vs_sft", "dpo_hh_hh", "dpo_hh", "sft_hh", "sft", HH),
    ("dpo_ultrafeedback_vs_sft", "dpo_ultrafeedback_alpaca", "dpo_ultrafeedback", "sft_alpaca", "sft", ALPACA),
    ("lora_vs_base_persona", "smollm2_lora_persona", "smollm2_lora", "smollm2_base_persona", "smollm2_base", PERSONA),
]
for comparison, x_file, x_name, y_file, y_name, prompts in COMPARISONS:
    run([sys.executable, "-m", "pa2.pairs", "--x", f"{ANSWERS}/{x_file}.json", "--name-x", x_name,
         "--y", f"{ANSWERS}/{y_file}.json", "--name-y", y_name, "--prompts", prompts,
         "--comparison", comparison, "--out", f"{OUT_DIR}/pairs/{comparison}.jsonl"],
        cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": f"{REPO_DIR}/src:{REPO_DIR}"})

## Statistics per model (stopping, loops, length)

In [ ]:
stats = {}
for path in sorted(glob.glob(f"{ANSWERS}/*.meta.json")):
    meta = json.load(open(path))
    stats[os.path.basename(path).replace(".meta.json", "")] = {**meta["stats"], "seconds": meta["seconds"]}
json.dump(stats, open(f"{OUT_DIR}/answer_stats.json", "w"), indent=2)
keys = ["count", "stopped_on_its_own", "hit_max_new_tokens", "repetition_loop", "mean_new_tokens", "mean_words"]
print(f"{'answers':<28}" + "".join(f"{k[:14]:>16}" for k in keys))
for name, row in stats.items():
    print(f"{name:<28}" + "".join(f"{row[k]:>16.3f}" for k in keys))

## Archive

In [ ]:
import zipfile


def zip_small_outputs(archive_path, root, skip_dirs=(), skip_suffixes=(".safetensors", ".jsonl.gz")):
    """Zip everything under root except checkpoints and bulky data, so the archive stays small."""
    count = 0
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for folder, dirs, files in os.walk(root):
            rel_folder = os.path.relpath(folder, root)
            dirs[:] = [d for d in dirs if os.path.normpath(os.path.join(rel_folder, d)) not in skip_dirs]
            for name in sorted(files):
                if name.endswith(skip_suffixes):
                    continue
                path = os.path.join(folder, name)
                archive.write(path, os.path.relpath(path, root))
                count += 1
    print(f"Saved {archive_path} ({count} files, {os.path.getsize(archive_path) / 1e6:.1f} MB)")


zip_small_outputs(f"{WORK_DIR}/task6_generation_outputs.zip", OUT_DIR)